In [4]:
!pip install -q copernicusmarine
import copernicusmarine

In [ ]:
copernicusmarine.login(check_credentials_valid=True)

INFO - 2026-09-29T16:46:38Z - Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.
INFO:copernicusmarine:Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.


True

In [6]:
import pandas as pd

df = pd.read_csv('/content/sample_data/harmful_algae_events.csv', low_memory=False)

valid_coords = (
    df["Latitude"].notna() & df["Longitude"].notna()
    & df["Latitude"].between(-90, 90)
    & df["Longitude"].between(-180, 180)
)
has_date = df[["Event Date", "Start Date"]].notna().any(axis=1)
not_freshwater = df["Freshwater"] == "No"
approved = df["Status"] == "Approved"

usable = df[valid_coords & has_date & not_freshwater & approved].copy()
usable["date"] = pd.to_datetime(usable["Event Date"].fillna(usable["Start Date"]))

sample = usable.sample(15, random_state=42)[["Event ID", "Country", "Latitude", "Longitude", "date"]]
sample

,Event ID,Country,Latitude,Longitude,date
4618,10562,FRENCH POLYNESIA,-16.994000,-142.251000,2014-09-21
3667,5269,FRENCH POLYNESIA,-17.178000,-150.244000,2016-12-29
7726,3871,CHINA,27.141091,120.470851,2008-05-01
12748,609,PORTUGAL,40.850000,-9.030000,1995-06-06
7215,11613,FRENCH POLYNESIA,-24.150000,-149.060000,2009-07-02
1896,9625,PHILIPPINES,8.244100,126.375700,2020-08-25
10094,4618,POLAND,54.930000,17.290000,2004-07-31
13616,167,FRANCE,49.080000,-2.050000,1989-10-05
770,14104,SPAIN,36.910000,-7.090000,2023-07-26
7624,8951,SPAIN,42.100000,-9.320000,2009-05-01


In [7]:
import copernicusmarine

ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
)

print(ds)
print("\nTime range:", ds.time.min().values, "to", ds.time.max().values)
print("\nVariables:", list(ds.data_vars))
print("\nDepth levels (first 10):", ds.depth.values[:10])
print("Lat/lon resolution:",
      float(ds.latitude[1] - ds.latitude[0]),
      float(ds.longitude[1] - ds.longitude[0]))

INFO - 2026-09-29T16:46:47Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-29T16:46:47Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


<xarray.Dataset> Size: 179TB
Dimensions:    (depth: 50, latitude: 2041, longitude: 4320, time: 12227)
Coordinates:
  * depth      (depth) float32 200B 0.494 1.541 2.646 ... 5.275e+03 5.728e+03
  * latitude   (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.83 89.92 90.0
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
  * time       (time) datetime64[ns] 98kB 1993-01-01 1993-01-02 ... 2026-06-23
Data variables:
    bottomT    (time, latitude, longitude) float64 862GB dask.array<chunksize=(50, 512, 2048), meta=np.ndarray>
    mlotst     (time, latitude, longitude) float64 862GB dask.array<chunksize=(50, 512, 2048), meta=np.ndarray>
    siconc     (time, latitude, longitude) float64 862GB dask.array<chunksize=(50, 512, 2048), meta=np.ndarray>
    sithick    (time, latitude, longitude) float64 862GB dask.array<chunksize=(50, 512, 2048), meta=np.ndarray>
    so         (time, depth, latitude, longitude) float64 43TB dask.array<chunksize=(50, 2, 512, 2048), meta

In [ ]:
records = []
for _, row in sample.iterrows():
    point = ds.sel(
        latitude=row["Latitude"],
        longitude=row["Longitude"],
        time=row["date"],
        depth=0,
        method="nearest"
    )
    record = {
        "Event ID": row["Event ID"],
        "Country": row["Country"],
        "Latitude": row["Latitude"],
        "Longitude": row["Longitude"],
        "date": row["date"],
    }
    for var in ds.data_vars:
        record[var] = point[var].values.item()
    records.append(record)

result = pd.DataFrame(records)
result

,Event ID,Country,Latitude,Longitude,date,bottomT,mlotst,siconc,sithick,so,thetao,uo,usi,vo,vsi,zos
0,10562,FRENCH POLYNESIA,-16.994000,-142.251000,2014-09-21,1.898587,25.025179,NaN,NaN,36.510819,26.534349,-0.056154,0.0,-0.195929,0.0,0.678121
1,5269,FRENCH POLYNESIA,-17.178000,-150.244000,2016-12-29,1.308237,12.359997,NaN,NaN,35.906552,28.858394,-0.039674,NaN,0.050050,NaN,0.672628
2,3871,CHINA,27.141091,120.470851,2008-05-01,21.238044,7.782220,NaN,NaN,27.329325,21.276864,0.023804,0.0,0.034791,0.0,0.193487
3,609,PORTUGAL,40.850000,-9.030000,1995-06-06,13.441176,11.444442,NaN,NaN,35.413678,15.285470,-0.063478,0.0,-0.104984,0.0,-0.379345
4,11613,FRENCH POLYNESIA,-24.150000,-149.060000,2009-07-02,1.029908,60.884429,NaN,NaN,35.351115,23.173894,-0.081790,0.0,0.086062,0.0,0.679952
5,9625,PHILIPPINES,8.244100,126.375700,2020-08-25,29.622333,10.376294,NaN,NaN,34.322642,30.329142,0.270394,NaN,-0.355235,NaN,0.804773
6,4618,POLAND,54.930000,17.290000,2004-07-31,12.433332,10.528886,NaN,NaN,7.098605,18.620289,-0.070803,0.0,-0.012818,0.0,-0.098880
7,167,FRANCE,49.080000,-2.050000,1989-10-05,8.563829,17.853329,NaN,NaN,35.068819,8.574816,-0.010376,0.0,0.006714,0.0,-0.388195
8,14104,SPAIN,36.910000,-7.090000,2023-07-26,15.155828,10.986664,NaN,NaN,36.266671,22.314737,0.083010,NaN,-0.013428,NaN,-0.300913
9,8951,SPAIN,42.100000,-9.320000,2009-05-01,12.297098,11.749627,NaN,NaN,35.532701,14.260048,-0.023804,0.0,-0.087283,0.0,-0.386975


In [ ]:
import copernicusmarine
import xarray as xr
import pandas as pd
from pathlib import Path
from datetime import timedelta
import time

tolerance = timedelta(hours=12)
records = []

for i, row in subset_range.iterrows():
    fname = f"cop_point_{row['Event ID']}.nc"

    if not Path(fname).exists():
        t0 = time.time()
        copernicusmarine.subset(
            dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
            variables=["thetao", "so", "bottomT", "mlotst", "zos", "uo", "vo"],
            minimum_longitude=row["Longitude"] - 0.5, maximum_longitude=row["Longitude"] + 0.5,
            minimum_latitude=row["Latitude"] - 0.5, maximum_latitude=row["Latitude"] + 0.5,
            start_datetime=row["date"] - tolerance,
            end_datetime=row["date"] + tolerance,
            minimum_depth=0, maximum_depth=1,
            coordinates_selection_method="nearest",
            output_filename=fname,
        )
        print(f"Event {row['Event ID']} downloaded in {time.time()-t0:.1f} sec")
    else:
        print(f"Event {row['Event ID']} already cached, skipping download")

    point_ds = xr.open_dataset(fname)
    point = point_ds.sel(latitude=row["Latitude"], longitude=row["Longitude"], time=row["date"], method="nearest")

    record = {"Event ID": row["Event ID"], "Country": row["Country"],
              "Latitude": row["Latitude"], "Longitude": row["Longitude"], "date": row["date"]}
    for var in point_ds.data_vars:
        record[var] = point[var].values.item()
    records.append(record)

full_result = pd.DataFrame(records)
full_result["label"] = 1  # these are all positive (bloom) events
full_result.to_csv("haedat_copernicus_features.csv", index=False)

Event 14793 already cached, skipping download
Event 14805 already cached, skipping download
Event 14794 already cached, skipping download
Event 14795 already cached, skipping download
Event 14796 already cached, skipping download
Event 14797 already cached, skipping download
Event 14798 already cached, skipping download
Event 14806 already cached, skipping download
Event 14801 already cached, skipping download
Event 14799 already cached, skipping download
Event 14802 already cached, skipping download
Event 14800 already cached, skipping download
Event 14803 already cached, skipping download
Event 14804 already cached, skipping download


INFO - 2026-09-29T18:14:10Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-29T18:14:10Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
WARNING - 2026-09-29T18:14:10Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
INFO - 2026-09-29T18:14:16Z - Total size of the download: 18.16 KB.
INFO:copernicusmarine:Total size of the download: 18.16 KB.


Event 14541 downloaded in 9.0 sec


INFO - 2026-09-29T18:14:19Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-29T18:14:19Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
WARNING - 2026-09-29T18:14:19Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
INFO - 2026-09-29T18:14:25Z - Total size of the download: 18.16 KB.
INFO:copernicusmarine:Total size of the download: 18.16 KB.


Event 14629 downloaded in 8.4 sec


...

Event 9304 downloaded in 8.4 sec


INFO - 2026-09-30T00:43:57Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-30T00:43:57Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
WARNING - 2026-09-30T00:43:57Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
INFO - 2026-09-30T00:44:02Z - Total size of the download: 18.16 KB.
INFO:copernicusmarine:Total size of the download: 18.16 KB.


Event 9303 downloaded in 8.4 sec


INFO - 2026-09-30T00:44:05Z - Selected dataset version: "202311"
INFO:copernicusmarine:Selected dataset version: "202311"
INFO - 2026-09-30T00:44:05Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"
WARNING - 2026-09-30T00:44:05Z - Some of your subset selection [0.0, 1.0] for the depth dimension exceed the dataset coordinates [0.49402499198913574, 5727.9169921875]
INFO - 2026-09-30T00:44:11Z - Total size of the download: 18.16 KB.
INFO:copernicusmarine:Total size of the download: 18.16 KB.


Event 9302 downloaded in 8.3 sec
